# Discrete-continuous choice: the Iskhakov et al. (2017) retirement model

This notebook illustrates the central numerical challenge of discrete-continuous
dynamic choice models on the deterministic retirement model of Iskhakov, Jørgensen,
Rust & Schjerning (2017), "The endogenous grid method for discrete-continuous dynamic
choice models with (or without) taste shocks", *Quantitative Economics* 8(2), 317–365,
[doi:10.3982/QE643](https://doi.org/10.3982/QE643). The model ships as
`lcm_examples.iskhakov_et_al_2017` and has a closed-form solution that pylcm's test
suite uses as an analytical oracle.

A worker chooses consumption and whether to keep working or to retire; retirement is
absorbing. The discrete retirement choice destroys the concavity of the value function
and produces the paper's signature **saw-tooth consumption function**, which we plot
below. The notebook currently solves the model by brute-force grid search; it will
grow into a side-by-side comparison with the DC-EGM solver.

In [ ]:
import jax.numpy as jnp
import plotly.graph_objects as go

from lcm_examples.iskhakov_et_al_2017 import get_model, get_params

## Solving the model

We use the paper's analytical-solution parametrization: log utility, work disutility
$\delta = 1$, wage $20$, discount factor $0.98$, zero interest. With `n_periods=6`
the agent makes decisions in five periods and is dead in the last one.

The consumption *policy* in the first period is read off a forward simulation that
starts one agent at each point of a dense wealth grid.

In [ ]:
PAPER_PARAMS = {
    "discount_factor": 0.98,
    "disutility_of_work": 1.0,
    "interest_rate": 0.0,
    "wage": 20.0,
}


def first_period_consumption(n_periods, wealth):
    model = get_model(n_periods)
    params = get_params(n_periods, **PAPER_PARAMS)
    result = model.simulate(
        params=params,
        initial_conditions={
            "age": jnp.full(wealth.size, model.ages.values[0]),
            "wealth": wealth,
            "regime_id": jnp.full(
                wealth.size, model.regime_names_to_ids["working_life"]
            ),
        },
        period_to_regime_to_V_arr=None,
        log_level="warning",
    )
    df = result.to_dataframe()
    return df.query("period == 0").sort_values("wealth")


WEALTH = jnp.linspace(1.0, 120.0, 600)
policy = first_period_consumption(n_periods=6, wealth=WEALTH)

## The saw-tooth consumption function

The first-period consumption function of a worker is not smooth — it jumps downward
at a sequence of wealth thresholds:

In [ ]:
fig = go.Figure(
    go.Scatter(
        x=policy["wealth"],
        y=policy["consumption"],
        mode="lines",
        line={"color": "steelblue"},
    )
)
fig.update_layout(
    title="First-period consumption of a worker (5 decision periods)",
    xaxis_title="wealth",
    yaxis_title="consumption",
    template="simple_white",
)
fig.show()

Each tooth corresponds to a different optimal retirement age. The pattern occurs
because as people retire later, their lifetime wealth increases, which changes the
optimal consumption path: at low current wealth the agent plans to work for many more
periods and consumes out of a large lifetime income; as wealth rises, the optimal
retirement date moves one period earlier at each threshold, lifetime labor income
drops by one year's wage, and consumption jumps down before resuming its smooth rise.

These *secondary kinks* propagate backward through the value function: the further the
agent is from the end of life, the more future retirement dates are in play and the
more teeth the consumption function has. Comparing first-period policies across
horizons makes this accumulation visible:

In [ ]:
fig = go.Figure()
for n_periods, color in zip(
    [3, 4, 5, 6], ["lightgray", "darkgray", "gray", "steelblue"], strict=True
):
    pol = first_period_consumption(n_periods=n_periods, wealth=WEALTH)
    fig.add_trace(
        go.Scatter(
            x=pol["wealth"],
            y=pol["consumption"],
            mode="lines",
            line={"color": color},
            name=f"{n_periods - 1} decision periods",
        )
    )
fig.update_layout(
    title="More remaining work-life, more teeth",
    xaxis_title="wealth",
    yaxis_title="first-period consumption",
    template="simple_white",
)
fig.show()

## Brute force vs DC-EGM

The figures above come from pylcm's brute-force solver: consumption is chosen from a
dense grid, so each tooth is resolved only up to the grid's resolution, and the cost
of the solve scales with the size of the consumption grid.

This is exactly the situation the DC-EGM algorithm of Iskhakov et al. (2017) is built
for: it inverts the Euler equation (no consumption grid at all), locates the kinks
exactly via an upper-envelope step, and handles the borrowing constraint in closed
form. Once pylcm's DC-EGM solver lands, this section will compare the two solvers'
accuracy and runtime on this model side by side.